In [ ]:
import numpy as np
from numpy import inf

In [ ]:
# Distance matrix representing the cost between 5 cities
d = np.array([[0, 10, 12, 11, 14],
              [10, 0, 13, 15, 8],
              [12, 13, 0, 9, 14],
              [11, 15, 9, 0, 16],
              [14, 8, 14, 16, 0]])

iteration = 100
n_ants = 5
n_citys = 5

In [ ]:
# Initialize Ant Colony Optimization (ACO) parameters
m = n_ants
n = n_citys
e = 0.1         # Evaporation rate (lower value retains pheromone trail memory longer)
alpha = 1       # Pheromone exponential weight factor
beta = 2        # Visibility/Heuristic exponential weight factor

In [ ]:
# Calculate the initial visibility matrix (inverse of distance)
with np.errstate(divide='ignore'):
    visibility = 1.0 / d
visibility[visibility == inf] = 0

In [ ]:
# Initialize the pheromone matrix uniformly
pheromne = 0.1 * np.ones((m, n))

In [ ]:
# Initialize routes array for all ants with shape (n_ants, n_cities + 1)
# Adding 1 to return to the starting city
rute = np.zeros((m, n + 1), dtype=int)
rute[:, 0] = 0

for ite in range(iteration):
    for i in range(m):
        temp_visibility = np.array(visibility)  # Copy visibility to track visited cities

        for j in range(n - 1):
            cur_loc = rute[i, j]
            temp_visibility[:, cur_loc] = 0  # Mark current city as visited

            # Compute probabilities using the ACO formula
            p_feature = np.power(pheromne[cur_loc, :], alpha)
            v_feature = np.power(temp_visibility[cur_loc, :], beta)

            combine_feature = p_feature * v_feature
            total = np.sum(combine_feature)

            if total == 0:
                unvisited = np.where(temp_visibility[cur_loc, :] > 0)[0]
                city = np.random.choice(unvisited) if len(unvisited) > 0 else cur_loc
            else:
                probs = combine_feature / total
                cum_prob = np.cumsum(probs)
                r = np.random.random_sample()
                city = np.nonzero(cum_prob > r)[0][0]

            rute[i, j + 1] = city

        # Assign the last unvisited city
        left = list(set(range(n)) - set(rute[i, :-2]))[0]
        rute[i, -2] = left

        # Complete the tour by returning to the start city
        rute[i, -1] = rute[i, 0]

    dist_cost = np.zeros((m, 1))

    # Calculate total tour distance for each ant
    for i in range(m):
        s = 0
        for j in range(n):
            s += d[rute[i, j], rute[i, j + 1]]
        dist_cost[i] = s

    dist_min_loc = np.argmin(dist_cost)
    dist_min_cost = dist_cost[dist_min_loc][0]
    best_route = rute[dist_min_loc, :]

    # Evaporate pheromones
    pheromne = (1 - e) * pheromne

    # Deposit new pheromones
    for i in range(m):
        dt = 1.0 / dist_cost[i].item()
        for j in range(n):
            from_city = rute[i, j]
            to_city = rute[i, j + 1]
            pheromne[from_city, to_city] += dt

print('Ant routes at the end of the final iteration:')
print(rute + 1)
print()
print('Best route found (1-based index):', best_route + 1)
print('Total cost of the best route:', int(dist_min_cost))

Ant routes at the end of the final iteration:
[[1 2 5 3 4 1]
 [1 4 3 2 5 1]
 [1 2 5 3 4 1]
 [1 2 5 3 4 1]
 [1 2 5 3 4 1]]

Best route found (1-based index): [1 2 5 3 4 1]
Total cost of the best route: 52


In [ ]:
import itertools

# Define starting city as City 1 (index 0)
start_city = 0
other_cities = [1, 2, 3, 4]  # Cities 2, 3, 4, 5 (0-based indexing)

# Generate all 24 possible permutations for the remaining cities
permutations = list(itertools.permutations(other_cities))

best_cost = float('inf')
best_tour = None

print(f"Evaluating all {len(permutations)} possible tours (Brute-Force Method):\n")
print(f"{'No':<4} | {'Tour (1-based)':<20} | {'Total Cost':<12}")
print("-" * 45)

for idx, perm in enumerate(permutations, 1):
    # Form a complete tour returning to the start city
    tour = [start_city] + list(perm) + [start_city]

    # Calculate the total tour distance
    cost = 0
    for i in range(len(tour) - 1):
        cost += d[tour[i], tour[i+1]]

    tour_display = " -> ".join(str(c + 1) for c in tour)
    print(f"{idx:<4} | {tour_display:<20} | {cost:<12}")

    if cost < best_cost:
        best_cost = cost
        best_tour = tour

print("-" * 45)
print(f"Optimal Tour from Brute-Force: {' -> '.join(str(c + 1) for c in best_tour)}")
print(f"Minimum Cost                 : {best_cost}")

Evaluating all 24 possible tours (Brute-Force Method):

No   | Tour (1-based)       | Total Cost  
---------------------------------------------
1    | 1 -> 2 -> 3 -> 4 -> 5 -> 1 | 62          
2    | 1 -> 2 -> 3 -> 5 -> 4 -> 1 | 64          
3    | 1 -> 2 -> 4 -> 3 -> 5 -> 1 | 62          
4    | 1 -> 2 -> 4 -> 5 -> 3 -> 1 | 67          
5    | 1 -> 2 -> 5 -> 3 -> 4 -> 1 | 52          
6    | 1 -> 2 -> 5 -> 4 -> 3 -> 1 | 55          
7    | 1 -> 3 -> 2 -> 4 -> 5 -> 1 | 70          
8    | 1 -> 3 -> 2 -> 5 -> 4 -> 1 | 60          
9    | 1 -> 3 -> 4 -> 2 -> 5 -> 1 | 58          
10   | 1 -> 3 -> 4 -> 5 -> 2 -> 1 | 55          
11   | 1 -> 3 -> 5 -> 2 -> 4 -> 1 | 60          
12   | 1 -> 3 -> 5 -> 4 -> 2 -> 1 | 67          
13   | 1 -> 4 -> 2 -> 3 -> 5 -> 1 | 67          
14   | 1 -> 4 -> 2 -> 5 -> 3 -> 1 | 60          
15   | 1 -> 4 -> 3 -> 2 -> 5 -> 1 | 55          
16   | 1 -> 4 -> 3 -> 5 -> 2 -> 1 | 52          
17   | 1 -> 4 -> 5 -> 2 -> 3 -> 1 | 60          
18   | 1 -> 4 -> 5 -> 